# OCR Extraction Pipeline — Setup & Test

Full litmus test for the extraction pipeline.

**Before starting:** Upload your sample PDFs/TIFFs to `/ocr/` using
Jupyter's file upload button.

This notebook will:
1. Check GPU and shared models
2. Load Qwen2.5-VL-7B directly
3. Load a sample document and check digital vs scanned
4. Run extraction
5. Inspect JSON output and experiment with formats
6. Process all pages
7. Launch Streamlit app

## 1. Check GPU and shared models

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os

os.environ["HF_HOME"] = "/models/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
    has_vlm = any("Qwen2.5-VL" in d for d in model_dirs)
    if has_vlm:
        print("\nQwen2.5-VL found.")
    else:
        print("\nWARNING: Qwen2.5-VL not found!")
        print("Run: python /models/provision_shared_models.py download Qwen/Qwen2.5-VL-7B-Instruct")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")

## 2. Load the model

Load Qwen2.5-VL-7B directly with transformers. Takes ~1-2 min.

In [ ]:
import time
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {MODEL_NAME}...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

elapsed = time.time() - t0
print(f"Model loaded in {elapsed:.1f}s")
print(f"Device: {model.device}")

In [ ]:
!pip install -q qwen-vl-utils

In [ ]:
from qwen_vl_utils import process_vision_info

def run_vlm(messages, max_tokens=4096):
    """Run inference on the loaded model."""
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def extract_text(text, prompt, max_tokens=4096):
    """Digital path: send extracted text to model."""
    full_prompt = f"{prompt}\n\n---\nDOCUMENT TEXT:\n---\n{text}"
    messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]
    return run_vlm(messages, max_tokens)

def extract_image(image, prompt, max_tokens=4096):
    """Scanned path: send image to model."""
    import base64, io
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"data:image/png;base64,{b64}"},
        {"type": "text", "text": prompt},
    ]}]
    return run_vlm(messages, max_tokens)

print("Helper functions ready.")

## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/sample.pdf")  # <-- UPDATE THIS

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Check digital vs scanned

In [ ]:
import fitz

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_info = []
for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    page_info.append({"page": i, "text": text, "has_text": has_text})
    status = "DIGITAL" if has_text else "SCANNED"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:150]}...")
    print()

digital = sum(1 for p in page_info if p["has_text"])
scanned = sum(1 for p in page_info if not p["has_text"])
print(f"Summary: {digital} digital, {scanned} scanned")
doc.close()

## 5. Run extraction on a single page

In [ ]:
from PIL import Image

PAGE_IDX = 0  # Change this to test different pages
info = page_info[PAGE_IDX]

# Prompt — try different ones from step 7!
PROMPT = """Extract all information from this document.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

t0 = time.time()

if info["has_text"]:
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path\n")
    result = extract_text(info["text"], PROMPT)
else:
    print(f"Page {PAGE_IDX+1}: Using SCANNED path\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    result = extract_image(img, PROMPT)

elapsed = time.time() - t0
print(f"\nExtraction took {elapsed:.1f}s")

## 6. Inspect the output

In [ ]:
import json

try:
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Not valid JSON: {e}\n")
    print("Raw output:")
    print(result)

## 7. Try different prompts

Copy one of these into `PROMPT` in step 5 and re-run.

In [ ]:
PROMPT_KV = """Extract all labeled data points from this document as key-value pairs.
Return a JSON object where keys are the field names and values are their values.
Preserve ALL values exactly. Output only valid JSON."""

PROMPT_BUDGET = """Extract budget information from this document.
Return a JSON object with: award_number, budget_period,
categories (array of {category, items: [{description, amount}], subtotal}),
total_direct, fa_rate, fa_base, total_indirect, total, cost_sharing, notes.
Preserve ALL dollar amounts exactly. Output only valid JSON."""

PROMPT_TERMS = """Extract terms and conditions from this document.
Return a JSON object with: document_title, effective_date,
sections (array of {number, title, text, subsections}),
definitions, references.
Preserve exact wording. Output only valid JSON."""

PROMPT_TEXT = """Extract all text from this document exactly as it appears.
Preserve the original reading order, line breaks, and structure.
Output only the extracted text."""

print("Copy one of these into PROMPT in step 5 and re-run.")
print("Available: PROMPT_KV, PROMPT_BUDGET, PROMPT_TERMS, PROMPT_TEXT")

## 8. Process all pages

In [ ]:
results = []
doc = fitz.open(str(DOC_PATH))

for info in page_info:
    t0 = time.time()
    if info["has_text"]:
        text = extract_text(info["text"], PROMPT)
        method = "text_extraction"
    else:
        mat = fitz.Matrix(2.0, 2.0)
        pix = doc[info["page"]].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        text = extract_image(img, PROMPT)
        method = "vlm_ocr"
    elapsed = time.time() - t0
    results.append({"page": info["page"] + 1, "method": method,
                    "elapsed_ms": round(elapsed * 1000, 1), "text": text})
    print(f"Page {info['page']+1}: {method} ({elapsed:.1f}s)")

doc.close()
print(f"\nDone. {len(results)} pages processed.")

In [ ]:
output = {
    "source_file": str(DOC_PATH),
    "total_pages": len(results),
    "digital_pages": sum(1 for r in results if r["method"] == "text_extraction"),
    "scanned_pages": sum(1 for r in results if r["method"] == "vlm_ocr"),
    "pages": results,
}
out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.json")
out_path.write_text(json.dumps(output, indent=2))
print(f"Saved to {out_path}")

## 9. Test Streamlit app

Launches the extraction server and Streamlit UI.
Access at: `https://<cluster-host>/<project>/ocr-setup/proxy/8501/`

**Requires a Custom URL tool on port 8501** in the workspace config.

In [ ]:
# TODO: Streamlit app currently requires a vLLM/Ollama endpoint
# (ocr_server.py talks to an OpenAI-compatible API, not the local model).
# To test Streamlit, start vLLM in a terminal first:
#
#   python -m vllm.entrypoints.openai.api_server \
#       --model Qwen/Qwen2.5-VL-7B-Instruct --dtype auto \
#       --max-model-len 8192 --limit-mm-per-prompt image=1
#
# Then run in another terminal:
#
#   cd /tmp/KohakuRAG_UI
#   LLM_BASE_URL=http://localhost:8000/v1 python ocr_app/scripts/ocr_server.py &
#   OCR_SERVICE_URL=http://localhost:8090 streamlit run ocr_app/app.py \
#       --server.port=8501 --server.address=0.0.0.0 --server.headless=true

print("See instructions above to test Streamlit.")
print("The notebook extraction (steps 2-8) uses the model directly — no vLLM needed.")
print("Streamlit requires vLLM because the extraction server talks to an API endpoint.")